# 04 — SSL embeddings vs. prosody baseline (length sweep + confound)

**The analytical payoff of the Feature-Exploration cycle.** Head-to-head between the
alignment-free **prosody baseline** and multilingual SSL embeddings (**XLS-R-300m**, headline)
on the lightly-expanded internet-radio corpus, measured with **method-agnostic proximity
metrics** (rhythm-class silhouette, within/between separation) and a **family-tree Mantel
test**, across analysis-window lengths **L ∈ {10s, 30s, full}** and an XLS-R layer sweep
**{8, 12, 16, 20, last}**, plus a **channel/station confound check** (design spec §3.3, §3.6,
§3.7, §4).

**Efficiency note.** SSL inference is the expensive step. We **embed once per length**: each
segment goes through XLS-R a single time with `output_hidden_states=True`, and every swept
layer is mean-pooled from that one forward pass. Per-segment, layer-tagged embeddings are
cached to `data/clip_embeddings_xlsr_<length>.parquet`, so re-runs are instant and the layer
sweep is essentially free.

**Coverage caveat (read first).** The corpus has **uneven coverage** — french 15/13, greek
15/15, spanish 13/13, polish 4/4, english 3/3, finnish 2/2, german 2/2, **italian 1/1**. Thin
languages (especially **italian = 1 clip / 1 station**) make several numbers low-confidence;
metrics degrade gracefully (NaN, not crash) on degenerate groups, and we surface those cases
honestly.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram

import torch
import librosa

from musiclang.config import DATA_DIR, TARGET_SAMPLE_RATE
from musiclang.features.prosody_acoustic import ProsodyAcousticExtractor
from musiclang.features.ssl_embedding import _load_model
from musiclang.pipeline import build_segment_features, clean_clip, segment_clip
from musiclang.features.aggregate import build_language_table
from musiclang.proximity.distance import standardize, distance_matrix, linkage_matrix, mds_2d
from musiclang.proximity.embedding import language_centroids
from musiclang.validation.proximity_agreement import (
    class_silhouette, within_between_separation, confound_report,
)
from musiclang.validation.family_tree import reference_distance_matrix, mantel_test
from musiclang.validation.typology import RHYTHM_CLASS

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

MODEL_ID = "facebook/wav2vec2-xls-r-300m"   # headline multilingual SSL model
LAYERS = [8, 12, 16, 20, -1]                 # XLS-R layer sweep (-1 = last)
LENGTHS = [None, 30.0, 10.0]                 # None == full clip; cheap-first so headline lands + caches early
MANTEL_PERMS = 10_000

manifest = pd.read_parquet(DATA_DIR / "clips_manifest.parquet")
languages = sorted(manifest["language"].unique())
print(f"{len(manifest)} clips, {len(languages)} languages: {languages}")

## 1. Coverage

Per-language clip and distinct-station counts straight from the manifest. This is the lens for
every number below: thin groups (italian, finnish, german, english) carry the largest
sampling noise.

In [ ]:
coverage = manifest.groupby("language").agg(
    clips=("clip_id", "size"), stations=("station_name", "nunique")
).sort_values("clips", ascending=False)
coverage

## 2. Scoring helpers + baseline feature cache

`score_distance` turns any per-language distance matrix into the four comparison metrics:
rhythm-class **silhouette**, **within/between gap**, and the genealogical **Mantel r / p**
(vs. the Glottolog reference over all 8 languages). The baseline path uses the scalar
`mean+std → standardize → euclidean` route; the SSL path uses `language_centroids → cosine`.

**Efficiency:** the prosody extractor (VAD + parselmouth) is the second-slowest step and the
notebook needs the per-length `(seg, feat)` table in three places (baseline distance, figures,
within-recording stability). `baseline_segment_features` builds it **once per length** (robust
to per-clip failures) and memoizes it, so it is never recomputed.

In [ ]:
_BASELINE_CACHE = {}

def baseline_segment_features(length_s):
    """Memoized robust (seg, feat) prosody table for a window length (built once)."""
    if length_s in _BASELINE_CACHE:
        return _BASELINE_CACHE[length_s]
    ex = ProsodyAcousticExtractor()
    segs, feats = [], []
    for i in range(len(manifest)):
        r1 = manifest.iloc[[i]]
        try:
            s, f = build_segment_features(r1, ex, length_s)
            if len(s):
                segs.append(s); feats.append(f)
        except Exception as exc:
            print(f"  [skip-base] {r1.iloc[0]['clip_id']}: {type(exc).__name__}: {exc}", flush=True)
    seg = pd.concat(segs); feat = pd.concat(feats)
    _BASELINE_CACHE[length_s] = (seg, feat)
    return seg, feat


def score_distance(dist_df, permutations=MANTEL_PERMS):
    """Rhythm-class silhouette + within/between gap + Mantel r/p vs. Glottolog reference."""
    labels = {l: RHYTHM_CLASS[l] for l in dist_df.index if l in RHYTHM_CLASS}
    ref = reference_distance_matrix(list(dist_df.index))
    r, p = mantel_test(dist_df, ref, permutations=permutations, seed=0)
    sep = within_between_separation(dist_df, labels)
    return {
        "silhouette": class_silhouette(dist_df, labels),
        "within_between_gap": sep["gap"],
        "mantel_r": r,
        "mantel_p": p,
    }


def baseline_distance(length_s):
    """Scalar prosody path: per-segment features -> per-language mean+std -> z-score -> euclidean."""
    seg, feat = baseline_segment_features(length_s)
    feat = feat.join(seg["language"])
    per_lang = {l: [row.drop(labels="language").to_dict() for _, row in g.iterrows()]
                for l, g in feat.groupby("language")}
    table = build_language_table(per_lang)
    return distance_matrix(standardize(table)), seg, feat

## 3. Embed once per length (all layers from one forward pass)

For each window length we iterate every segment **once**, run XLS-R a **single** time with
`output_hidden_states=True`, and mean-pool **each** swept layer from that one pass. The result
is cached per length to `data/clip_embeddings_xlsr_<length>.parquet` (one row per segment;
columns are layer-tagged `l{layer}_emb_NNN` plus full provenance). A re-run loads the cache and
skips inference entirely. This is the spec's efficiency fix over calling the model once per
layer (which would re-run XLS-R 5× per length).

> First execution downloads XLS-R-300m (~1.2 GB) and runs ~hundreds of CPU forward passes — the
> slow step. Everything after reads the cache.

In [ ]:
PROV_COLS = ["clip_id", "language", "station_name", "window_index", "start_s", "length_s"]


def embed_all_layers(manifest, length_s, layers=LAYERS, model_id=MODEL_ID, device="cpu"):
    """ONE forward pass per segment; mean-pool every layer in `layers`. Cached per length."""
    short = "xlsr" if "xls-r" in model_id else model_id.split("/")[-1]
    cache = DATA_DIR / f"clip_embeddings_{short}_{length_s}.parquet"
    if cache.exists():
        df = pd.read_parquet(cache)
        if all(any(c.startswith(f"l{ly}_emb_") for c in df.columns) for ly in layers):
            return df  # cache hit covers every requested layer
    feat, model = _load_model(model_id, device)
    model_sr = int(getattr(feat, "sampling_rate", TARGET_SAMPLE_RATE))
    rows, skipped = [], []
    n = len(manifest)
    for ci, (_, r) in enumerate(manifest.iterrows()):
        try:
            signal = clean_clip(r["path"])
            if len(signal) == 0:
                print(f"  [empty] {r['clip_id']}", flush=True)
                continue
            for meta, samples in segment_clip(
                r["clip_id"], r["language"], r["station_name"], signal, TARGET_SAMPLE_RATE, length_s
            ):
                sig = samples
                if model_sr != TARGET_SAMPLE_RATE:
                    sig = librosa.resample(sig.astype(np.float32), orig_sr=TARGET_SAMPLE_RATE, target_sr=model_sr)
                inputs = feat(sig, sampling_rate=model_sr, return_tensors="pt")
                inputs = {k: v.to(device) for k, v in inputs.items()}
                with torch.no_grad():
                    out = model(**inputs, output_hidden_states=True)
                row = dict(meta)  # provenance
                for ly in layers:
                    pooled = out.hidden_states[ly].squeeze(0).mean(dim=0).detach().cpu().numpy().astype(float)
                    for i, x in enumerate(pooled):
                        row[f"l{ly}_emb_{i:03d}"] = float(x)
                rows.append(row)
                del out
        except Exception as exc:
            skipped.append(r["clip_id"])
            print(f"  [skip-embed] {r['clip_id']}: {type(exc).__name__}: {exc}", flush=True)
        print(f"  embed L={length_s} {ci + 1}/{n} ({r['clip_id']})", flush=True)
    if skipped:
        print(f"  embed L={length_s}: skipped {len(skipped)}: {skipped}", flush=True)
    df = pd.DataFrame(rows).set_index("segment_id")
    df.to_parquet(cache)
    return df


def layer_emb_df(all_emb, layer):
    """Slice one layer's columns -> emb_df (emb_* cols + provenance) for language_centroids."""
    cols = [c for c in all_emb.columns if c.startswith(f"l{layer}_emb_")]
    out = all_emb[cols].copy()
    out.columns = [c.replace(f"l{layer}_", "") for c in cols]
    for p in ("language", "clip_id", "station_name"):
        out[p] = all_emb[p]
    return out


def ssl_distance_from(all_emb, layer):
    """Per-language cosine distance from L2-normalized per-recording centroids, one layer."""
    cent = language_centroids(layer_emb_df(all_emb, layer))  # L2-norm + per-recording weighting
    return distance_matrix(cent, metric="cosine")

## 4. Length × method × layer comparison table

The headline result. One row for the prosody baseline and one per XLS-R layer
{8, 12, 16, 20, last}, across L ∈ {10s, 30s, full}. SSL inference happens once per length
(cached); the layer rows are free slices of that cache.

In [ ]:
rows = []
emb_by_length = {}
for length_s in LENGTHS:
    bd, _, _ = baseline_distance(length_s)
    rows.append({"method": "prosody_baseline", "length_s": length_s, "layer": None, **score_distance(bd)})
    all_emb = embed_all_layers(manifest, length_s)   # ONE forward pass per segment, all layers
    emb_by_length[length_s] = all_emb
    for layer in LAYERS:
        sd = ssl_distance_from(all_emb, layer)
        rows.append({"method": "xlsr", "length_s": length_s, "layer": layer, **score_distance(sd)})

results = pd.DataFrame(rows)
results["length_lbl"] = results["length_s"].map(lambda x: "full" if x is None else f"{x:g}s")
results.sort_values(["method", "length_s", "silhouette"], ascending=[True, True, False])

### Best configurations side by side

Best XLS-R config by silhouette and by Mantel r, against the best baseline, so the
"does SSL win?" question is answered directly from the numbers.

In [ ]:
xlsr = results[results.method == "xlsr"]
base = results[results.method == "prosody_baseline"]

best_sil = xlsr.loc[xlsr["silhouette"].idxmax()] if xlsr["silhouette"].notna().any() else xlsr.iloc[0]
best_mantel = xlsr.loc[xlsr["mantel_r"].idxmax()] if xlsr["mantel_r"].notna().any() else xlsr.iloc[0]
base_best_sil = base.loc[base["silhouette"].idxmax()] if base["silhouette"].notna().any() else base.iloc[0]
base_best_mantel = base.loc[base["mantel_r"].idxmax()] if base["mantel_r"].notna().any() else base.iloc[0]

summary = pd.DataFrame([
    {"config": "baseline (best silhouette)", **base_best_sil[["length_lbl", "layer", "silhouette", "within_between_gap", "mantel_r", "mantel_p"]].to_dict()},
    {"config": "baseline (best Mantel r)",   **base_best_mantel[["length_lbl", "layer", "silhouette", "within_between_gap", "mantel_r", "mantel_p"]].to_dict()},
    {"config": "XLS-R (best silhouette)",    **best_sil[["length_lbl", "layer", "silhouette", "within_between_gap", "mantel_r", "mantel_p"]].to_dict()},
    {"config": "XLS-R (best Mantel r)",      **best_mantel[["length_lbl", "layer", "silhouette", "within_between_gap", "mantel_r", "mantel_p"]].to_dict()},
])
print("XLS-R beats baseline on silhouette:",
      bool(best_sil["silhouette"] > base_best_sil["silhouette"]),
      f'({best_sil["silhouette"]:.3f} vs {base_best_sil["silhouette"]:.3f})')
print("XLS-R beats baseline on Mantel r:  ",
      bool(best_mantel["mantel_r"] > base_best_mantel["mantel_r"]),
      f'({best_mantel["mantel_r"]:.3f} vs {base_best_mantel["mantel_r"]:.3f})')
summary

## 5. Channel / station confound check (spec §3.6)

For the best XLS-R config we build a **segment-level** cosine distance matrix from the
L2-normalized per-segment embeddings, then ask whether the geometry clusters by **language**
or by **station/channel**. If station separation rivals/exceeds language separation, XLS-R is
partly measuring the channel, not the language's sound — a decision-relevant outcome.

Station silhouette is expected to be degenerate (NaN) here: most stations contribute a single
clip, so "within-station" structure barely exists; `class_silhouette` returns NaN on such
degenerate labellings rather than crashing. We surface it honestly.

In [ ]:
best = best_sil  # best XLS-R config by rhythm-class silhouette
best_len = best["length_s"]
best_layer = int(best["layer"])
print(f"Best XLS-R config: layer {best_layer}, length {best['length_lbl']}, silhouette {best['silhouette']:.3f}")

all_emb = emb_by_length[best_len]
edf = layer_emb_df(all_emb, best_layer)
emb_cols = [c for c in edf.columns if c.startswith("emb_")]
x = edf[emb_cols].to_numpy(dtype=float)
norms = np.linalg.norm(x, axis=1, keepdims=True)
norms[norms == 0] = 1.0
unit = pd.DataFrame(x / norms, index=edf.index, columns=emb_cols)
seg_dist = distance_matrix(unit, metric="cosine")

lang_labels = all_emb["language"].to_dict()
station_labels = all_emb["station_name"].to_dict()
n_stations = all_emb["station_name"].nunique()
singleton_stations = (all_emb.groupby("station_name").size() == 1).sum()
print(f"{len(all_emb)} segments, {n_stations} stations ({singleton_stations} singleton-segment stations)")

confound = confound_report(seg_dist, lang_labels, station_labels)
print()
for k, v in confound.items():
    print(f"  {k:22s} {v}")
pd.Series(confound, name="value").to_frame()

## 6. Figures — dendrogram + MDS colored by rhythm class

Prosody baseline (full length) and the best XLS-R config. Points are colored by rhythm class
(stress / syllable / intermediate) so visual clustering can be read against the typology.

In [ ]:
CLASS_COLORS = {"stress": "tab:red", "syllable": "tab:blue", "intermediate": "tab:green"}

def plot_method(dist_df, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
    dendrogram(linkage_matrix(dist_df), labels=list(dist_df.index), ax=ax1)
    ax1.set_title(f"{title} — dendrogram")
    ax1.tick_params(axis="x", rotation=45)
    coords = mds_2d(dist_df)
    for lang, row in coords.iterrows():
        cls = RHYTHM_CLASS.get(lang, "?")
        ax2.scatter(row["mds_x"], row["mds_y"], color=CLASS_COLORS.get(cls, "gray"), s=80)
        ax2.annotate(lang, (row["mds_x"], row["mds_y"]), xytext=(4, 4), textcoords="offset points")
    handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=9, label=cls)
               for cls, c in CLASS_COLORS.items()]
    ax2.legend(handles=handles, title="rhythm class", loc="best", fontsize=8)
    ax2.set_title(f"{title} — MDS (colored by rhythm class)")
    plt.tight_layout()
    plt.show()

plot_method(baseline_distance(None)[0], "prosody baseline (full)")
plot_method(ssl_distance_from(emb_by_length[best_len], best_layer),
            f"XLS-R layer {best_layer} ({best['length_lbl']})")

## 7. Within-recording stability vs. length (spec §3.7)

A robustness signal that isolates short-sample noise from speaker/station variation: for
L ∈ {10s, 30s}, the std of a baseline rhythm metric (`npvi_v`) across each recording's windows,
averaged over recordings. Lower = the metric is more stable within a recording at that length.
(Full length is a single window per clip, so within-recording spread is undefined there.)

In [ ]:
def within_recording_stability(length_s, metric="npvi_v"):
    """Mean over recordings of the per-recording std of `metric` across that recording's windows."""
    seg, feat = baseline_segment_features(length_s)   # reuses the per-length cache
    feat = feat.join(seg["clip_id"])
    per_rec_std = feat.groupby("clip_id")[metric].std()   # NaN for single-window recordings
    return float(per_rec_std.mean())

stability = pd.DataFrame({
    "length_s": [10.0, 30.0],
    "mean_within_recording_npvi_v_std": [within_recording_stability(10.0), within_recording_stability(30.0)],
})
stability

## 8. Summary — read from the numbers above

The cell below assembles the verdict from the computed results (not hard-coded), so it stays
honest if numbers shift on re-run. A later task writes the formal decision doc from these.

In [ ]:
lines = []
sil_win = bool(best_sil["silhouette"] > base_best_sil["silhouette"])
man_win = bool(best_mantel["mantel_r"] > base_best_mantel["mantel_r"])
lang_dom = (
    not math.isnan(confound["language_silhouette"])
    and (math.isnan(confound["station_silhouette"]) or confound["language_silhouette"] > confound["station_silhouette"])
    and confound["language_gap"] > confound["station_gap"]
)

lines.append("### Verdict\n")
lines.append(f"- **Silhouette:** XLS-R best = {best_sil['silhouette']:.3f} "
             f"(layer {int(best_sil['layer'])}, {best_sil['length_lbl']}) vs baseline best "
             f"{base_best_sil['silhouette']:.3f} ({base_best_sil['length_lbl']}) -> "
             f"XLS-R **{'WINS' if sil_win else 'does NOT win'}** on rhythm-class separation.")
lines.append(f"- **Mantel r (family tree):** XLS-R best = {best_mantel['mantel_r']:.3f} "
             f"(p={best_mantel['mantel_p']:.3f}, layer {int(best_mantel['layer'])}, {best_mantel['length_lbl']}) "
             f"vs baseline best {base_best_mantel['mantel_r']:.3f} (p={base_best_mantel['mantel_p']:.3f}) -> "
             f"XLS-R **{'WINS' if man_win else 'does NOT win'}** on genealogical agreement.")
lines.append(f"- **Best layer / length:** silhouette favors layer {int(best_sil['layer'])} @ {best_sil['length_lbl']}; "
             f"Mantel favors layer {int(best_mantel['layer'])} @ {best_mantel['length_lbl']}.")
lines.append(f"- **Confound:** language silhouette = {confound['language_silhouette']:.3f}, "
             f"station silhouette = {confound['station_silhouette']} "
             f"(NaN = degenerate / singleton stations); language gap = {confound['language_gap']:.4f}, "
             f"station gap = {confound['station_gap']:.4f} -> language clustering "
             f"**{'dominates' if lang_dom else 'does NOT clearly dominate'}** station clustering.")
lines.append(f"- **Within-recording stability:** mean per-recording npvi_v std = "
             f"{stability.iloc[0,1]:.3f} (10s) vs {stability.iloc[1,1]:.3f} (30s) — "
             f"{'longer windows are more stable' if stability.iloc[1,1] < stability.iloc[0,1] else 'shorter windows are not less stable'}.")
lines.append("")
lines.append("### Coverage caveats")
lines.append("- Uneven coverage: french/greek/spanish are well-sampled; **italian = 1 clip / 1 station**, "
             "finnish/german/english are thin (2-3 clips). Per-language centroids and the rhythm-class "
             "silhouette for these languages are high-variance; treat single-config differences cautiously.")
lines.append("- Station silhouette is NaN/degenerate because most stations contribute one clip — the "
             "confound check leans on the language-vs-station **gap** comparison rather than station silhouette.")
lines.append("- The Glottolog reference is a coarse depth-based 8x8 tree; Mantel r is directional evidence, not a precise effect size.")

from IPython.display import Markdown
Markdown("\n".join(lines))